# HRH: full pipeline

Runtime → Change runtime type → **GPU**. Everything is written to Google Drive, so an interrupted
cell can simply be rerun: finished subjects and models are skipped.

1. **Development run**: brain RDMs, features for all models, evaluation.
2. **Law**: brain-free proxies, fit and freeze the law, predictions for the held-out models.
3. **Registration**: the predictions are registered (OSF) before any held-out brain analysis.
4. **Confirmatory test**: evaluation including the held-out models.

In [ ]:
# Settings
REPO = "https://github.com/LegallyCoder/hrh"
OUT  = "/content/drive/MyDrive/hrh_results"   # persistent

from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Code + dependencies (~1 min)
%cd /content
!rm -rf hrh && git clone -q $REPO hrh
%cd /content/hrh
!pip -q install -e ".[models,dev]"
!pytest -q
!git log --oneline -1

## 1. Development run
Skip any cell whose outputs are already on Drive.

In [ ]:
!python run_hrh.py brain --out $OUT                 # ~1 h: EEG download, RDMs, ceilings

In [ ]:
!python run_hrh.py features --out $OUT --clean-cache  # all models incl. held-out (Grams only, no brain data)

In [ ]:
!python run_hrh.py eval --out $OUT --paper $OUT/paper  # held-out models are excluded until registered

## 2. Law (brain-free proxies → fit → held-out predictions)
`proxies` uses only Gram matrices, the images and human category labels. `law` is fit on the
development models. `predict` writes `registration/predictions_heldout.json` and its SHA-256.

In [ ]:
!python run_hrh.py proxies --out $OUT
!python run_hrh.py law     --out $OUT
!python run_hrh.py predict --out $OUT --reg-dir $OUT/registration

In [ ]:
from google.colab import files
files.download(f"{OUT}/registration/predictions_heldout.json")
files.download(f"{OUT}/registration/predictions_heldout.sha256")

## 3. Registration
Register `predictions_heldout.json` on OSF and commit both files to `registration/` in the
repository. Then rerun the **Code + dependencies** cell so the clone contains them.

## 4. Confirmatory test

In [ ]:
!python run_hrh.py eval --out $OUT --paper $OUT/paper   # now includes law_test

In [ ]:
# Package the results (a few MB)
!cd $OUT && zip -qr /content/hrh_results.zip paper/generated paper/figures summary_*.json models_*.csv proxies_*.json law.json
from google.colab import files
files.download('/content/hrh_results.zip')